# Parameter-Efficient Fine-Tuning of BERT for Text Classification using QLORA

## Objective:

The aim of this project is to apply parameter-efficient fine-tuning (PEFT) techniques,
specifically QLoRA (Quantized Low-Rank Adaptation), to fine-tune a BERT-based language
model for a text classification task, while significantly reducing the computational and memory
requirements typically associated with fine-tuning large language models (LLMs).

## Background and Motivation:

Large Language Models (LLMs) like BERT have revolutionized NLP tasks such as sentiment analysis,
question answering, and classification. However, full fine-tuning of such models is computationally
expensive and requires vast resources. To make fine-tuning feasible for resource-constrained
environments (e.g., personal machines or small servers), researchers have developed PEFT
methods like LoRA and its optimized variant QLoRA.

QLoRA uses quantization (e.g., 4-bit quantization) and low-rank adaptations to fine-tune only a small
portion of the model, reducing both memory and time costs without significantly compromising
accuracy.

https://medium.com/@rajratangulab.more/fine-tuning-bert-for-text-classification-using-hugging-face-transformers-685c132d185d

In [1]:
from transformers import TrainingArguments, set_seed, BertTokenizer, BertForSequenceClassification, BitsAndBytesConfig

from datasets import load_dataset
from transformers import Trainer
from peft import get_peft_model, LoraConfig, TaskType
import torch

seed = 42
set_seed(seed)

c:\Users\raad\Desktop\GenAI-Virtual-Environment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tasks:

### 1. Data Preparation and Tokenization:

-   Load a binary classification dataset (e.g., Amazon Polarity or IMDb). Feel free to pick a
dataset, our suggestion would be to
use https://huggingface.co/datasets/dipanjanS/imdb_sentiment_finetune_dataset20k this
dataset

-   Preprocess the data.

-   Split the dataset into train and test sets.

In [2]:
dataset_name = "dipanjanS/imdb_sentiment_finetune_dataset20k"

dataset = load_dataset(dataset_name)

dataset = dataset.rename_columns(
    {
        "review": "text",
        "sentiment": "label",
    }
)

In [3]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

In [4]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)

### 2. Model Setup:

-   Load a pre-trained bert-base-uncased model using HuggingFace Transformers.

-   Apply 4-bit quantization using bitsandbytes.

-   Integrate QLoRA adapters with LoRA configuration.

In [5]:
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 3. Training the Model:

-   Configure a Trainer using HuggingFace's transformers. Trainer API.

-   Train the model using parameter-efficient strategies.

-   Monitor evaluation metrics such as loss and accuracy. Try logging training to Weights
and biases.

In [6]:
model.train()
model = model.to("cuda")

In [7]:
output_dir = "./results"
per_device_train_batch_size = 1
per_device_eval_batch_size = 1
gradient_accumulation_steps = 8
logging_steps = 5
learning_rate = 5.e-4
max_grad_norm = 1.0
max_steps = 250
num_train_epochs=3
warmup_ratio = 0.1
lr_scheduler_type = "cosine"

training_args = TrainingArguments(
    output_dir=output_dir,
    # per_device_train_batch_size=per_device_train_batch_size,
    # per_device_eval_batch_size=per_device_eval_batch_size,
    # gradient_accumulation_steps=gradient_accumulation_steps,
    save_strategy="best",
    metric_for_best_model="loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    eval_strategy="epoch",
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    max_grad_norm=max_grad_norm,
    weight_decay=0.1,
    warmup_ratio=warmup_ratio,
    lr_scheduler_type=lr_scheduler_type,
    bf16=True,
    # report_to=["tensorboard", "wandb"],
    # hub_private_repo=True,
    # push_to_hub=True,
    num_train_epochs=num_train_epochs,
    # gradient_checkpointing=True,
    # gradient_checkpointing_kwargs={"use_reentrant": False}
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets["validation"]
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.715500,0.693971
2,0.685300,0.694928
3,0.669200,0.693408


TrainOutput(global_step=3000, training_loss=0.7073361129760742, metrics={'train_runtime': 187.6575, 'train_samples_per_second': 127.893, 'train_steps_per_second': 15.987, 'total_flos': 6314665328640000.0, 'train_loss': 0.7073361129760742, 'epoch': 3.0})

### 4. Evaluation:

-   Evaluate the fine-tuned model on the test set.

-   Compare performance in terms of classification accuracy and memory/parameter
efficiency.

In [12]:
best_model = BertForSequenceClassification.from_pretrained("./results/checkpoint-3000")
best_model = best_model.to("cuda")

In [13]:
predictions = []

best_model.eval()

for row in tokenized_datasets["test"]:
    inputs = {
    "input_ids": torch.tensor([row["input_ids"]]).to(best_model.device),
    "attention_mask": torch.tensor([row["attention_mask"]]).to(best_model.device)
}

    with torch.no_grad():
        outputs = best_model(**inputs)
        logits = outputs.logits
    
        prediction = torch.argmax(logits, dim=-1).item()

    predictions.append(prediction)

In [14]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

labels = list(tokenized_datasets["test"]["label"])
    
acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    
print(f"accuracy: {acc: .1%}, f1: {f1: .1%}")


accuracy:  51.2%, f1:  34.7%


### 5. Analysis and Interpretation:

-   Analyze the number of trainable parameters before and after applying QLoRA.

-   Demonstrate the memory efficiency gained by using QLoRA instead of full fine-tuning.